# Point & Ask prompt eval

Runs the classify -> score -> branch pipeline (`backend.point_and_ask`) against the synthetic fixture images in `eval/point_and_ask_images/`, then eyeballs explain-branch output quality on a couple of legitimate cases. Outputs are saved in this notebook on run.

In [1]:
import json
import os
import sys
from pathlib import Path

if not (Path.cwd() / "pyproject.toml").exists():
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

from backend.point_and_ask import classify_image, decide_branch, explain_image, score_risk

CASES_PATH = Path.cwd() / "eval" / "point_and_ask_eval_cases.json"
cases = json.loads(CASES_PATH.read_text())
len(cases)

10

In [2]:
passed = 0
for i, case in enumerate(cases, start=1):
    image_bytes = (Path.cwd() / "eval" / case["image"]).read_bytes()
    result = classify_image(image_bytes, "image/png")
    risk_level = score_risk(result)
    classification = decide_branch(result, risk_level)

    ok = classification == case["expected_classification"]
    passed += ok
    status = "PASS" if ok else "FAIL"
    print(
        f"[{status}] case {i} ({case['image']}): got={classification!r} "
        f"(risk={risk_level!r}) expected={case['expected_classification']!r}"
    )
print(f"\n{passed}/{len(cases)} passed")

2026-07-29 21:13:42.146 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


[PASS] case 1 (point_and_ask_images/case_01_scam_bank.png): got='scam' (risk='high') expected='scam'


[PASS] case 2 (point_and_ask_images/case_02_scam_prize.png): got='scam' (risk='medium') expected='scam'


[PASS] case 3 (point_and_ask_images/case_03_scam_tax.png): got='scam' (risk='high') expected='scam'


[PASS] case 4 (point_and_ask_images/case_04_legit_checkin.png): got='explain' (risk='low') expected='explain'


...

[PASS] case 6 (point_and_ask_images/case_07_scam_parcel.png): got='scam' (risk='medium') expected='scam'


[PASS] case 7 (point_and_ask_images/case_08_legit_medical.png): got='explain' (risk='low') expected='explain'


[PASS] case 8 (point_and_ask_images/case_09_scam_grandchild.png): got='scam' (risk='high') expected='scam'


[PASS] case 9 (point_and_ask_images/case_10_legit_community.png): got='explain' (risk='low') expected='explain'


[PASS] case 10 (point_and_ask_images/case_06_unreadable.png): got='unclear' (risk='low') expected='unclear'

10/10 passed


## Explain-branch quality check

Not pass/fail — eyeballing the actual explanation + translation text for a couple of legitimate documents, since the automated eval above only checks routing correctness.

In [3]:
for image_name in ["case_05_legit_passport.png", "case_08_legit_medical.png"]:
    image_bytes = (Path.cwd() / "eval" / "point_and_ask_images" / image_name).read_bytes()
    summary, translation = explain_image(image_bytes, "image/png")
    print(f"=== {image_name} ===")
    print(f"SUMMARY:\n{summary}\n")
    print(f"TRANSLATION:\n{translation}")
    print("-" * 60)

=== case_05_legit_passport.png ===
SUMMARY:
This is a reminder letter about your passport. It says your passport will stop being valid (expire) next month. You should go to the nearest ICA office (Immigration & Checkpoints Authority) to renew it before that happens. Since this involves a deadline and official paperwork, it would be a good idea to ask a family member to help you plan the visit or gather the documents needed.

TRANSLATION:
这是一份关于您护照的提醒信。信中说您的护照将在下个月到期（失效）。您应该在护照到期之前，前往最近的移民与关卡局（ICA）办事处办理更新手续。由于这涉及到期限和官方文件，建议您请家人协助安排办理时间或准备所需的证件。
------------------------------------------------------------


=== case_08_legit_medical.png ===
SUMMARY:
This is a reminder for a doctor's appointment. You have a visit scheduled with Dr. Lim on Tuesday at 3pm, at Raffles Medical clinic. Please remember to bring your NRIC (your identity card) with you to the appointment. It may help to have a family member confirm the exact date and help arrange transportation if needed.

TRANSLATION:
这是一份医生预约提醒。您与林医生（Dr. Lim）有一个预约，时间是星期二下午3点，地点在莱佛士医疗（Raffles Medical）诊所。请记得带上您的身份证（NRIC）去看诊。建议请家人帮忙确认具体日期，并在需要时协助安排交通。
------------------------------------------------------------
